# 🍔 Food Delivery Product Funnel & Retention Analytics
### End-to-End Product Analytics Teardown & Experimentation Framework
**Tech Stack**: SQL · Python · Power BI / Tableau Schemas  
**Empirical Data**: 41,176 Sessions, 4,200 Orders, 500 Longitudinal Customer Profiles
---
## 1. Executive Context & The Core Business Question
> **The Business Question**: *Where are users dropping off before placing an order, and what might encourage them to return?*

In high-frequency marketplace applications like food delivery, top-of-funnel exploration is inexpensive, but **late-funnel drop-off is disastrous**. Every user who builds a cart and enters checkout has already spent minutes browsing menus, choosing dishes, and deciding to dine. When that user leaves without placing an order, customer acquisition costs (CAC) and intent are incinerated.

This notebook tracks the end-to-end journey from **App Open → Restaurant Discovery → Add to Cart → Initiate Checkout → Successful Order**, identifies the **biggest drop-off bottleneck**, diagnoses root causes, and designs a testable product intervention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Load raw datasets
df_users = pd.read_csv("../data/raw/users.csv")
df_sessions = pd.read_csv("../data/raw/sessions.csv")
df_events = pd.read_csv("../data/raw/events.csv")
df_orders = pd.read_csv("../data/raw/orders.csv")

print(f"Users Loaded:    {len(df_users):,}")
print(f"Sessions Loaded: {len(df_sessions):,}")
print(f"Events Loaded:   {len(df_events):,}")
print(f"Orders Loaded:   {len(df_orders):,}")

## 2. Overall Funnel Conversion & Drop-off Quantification
We calculate pass-through volumes, step-over-step conversions, and cumulative conversion rates from initial App Open.

In [ ]:
df_funnel = pd.read_csv("../data/processed/funnel_summary.csv")
df_funnel

In [ ]:
# Visualize the Funnel Waterfall
stages = [s.split('. ')[1] for s in df_funnel['stage_name']]
sessions = df_funnel['sessions']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages, sessions, color=['#3B82F6', '#60A5FA', '#93C5FD', '#F59E0B', '#10B981'], edgecolor='#0F172A')
ax.set_ylabel('Session Volume', fontweight='bold')
ax.set_title('Food Delivery Conversion Funnel: App Open to Order Success', fontsize=13, fontweight='bold')

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 800, f'{int(h):,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 🚨 Key Takeaway: The Checkout Bottleneck
- **Stage 4 ➔ Stage 5 Drop-off is 70.63%**: Out of 14,298 sessions that reached checkout, **10,098 abandoned before placing an order**.
- End-to-end conversion is **10.20%**.

## 3. Segmentation: Platform, City, Segment & Promo Impact
How do funnel drop-offs differ across strategic operational dimensions?

In [ ]:
df_seg = pd.read_csv("../data/processed/segment_funnel.csv")
df_seg[["dimension_type", "dimension_value", "total_sessions", "checkout_to_order_pct", "end_to_end_conversion_pct"]]

## 4. Cohort Retention: D1, D7, D14, D30 Decay
We evaluate longitudinal engagement across monthly signup cohorts.

In [ ]:
df_cohort = pd.read_csv("../data/processed/cohort_retention.csv")
active_cohorts = df_cohort[df_cohort["cohort_month"] >= "2023-12"].copy()
heatmap_matrix = active_cohorts.set_index("cohort_month")[["d1_session_retention_pct", "d7_session_retention_pct", "d14_session_retention_pct", "d30_session_retention_pct"]]
heatmap_matrix.columns = ["Day 1", "Day 7", "Day 14", "Day 30"]

plt.figure(figsize=(8, 4))
sns.heatmap(heatmap_matrix, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=1)
plt.title("Cohort Retention Matrix (% Active Sessions)", fontweight='bold')
plt.show()

## 5. Root Cause Investigation: Why Did 10,098 Users Abandon at Checkout?

In [ ]:
# Primary failure drivers at checkout
reasons = ["ETA Friction (>45m delay)", "Fee Shock Abandonment", "Price Comparison / Hesitation", "Payment Gateway Timeout", "Promo Restriction Barrier"]
counts = [3193, 2595, 1907, 1873, 530]

plt.figure(figsize=(8, 4))
plt.barh(reasons, counts, color=['#EF4444', '#F97316', '#FBBF24', '#6366F1', '#38BDF8'], edgecolor='#0F172A')
plt.gca().invert_yaxis()
plt.title('Why 10,098 Qualified Buyers Abandoned at Checkout', fontweight='bold')
plt.xlabel('Sessions Lost')
plt.show()

## 6. Proposed Product Intervention: Project Clarity
### The Fee-Free Milestone Engine & Dynamic All-In Pricing
1. **All-In Pricing Preview on Menus**: Reveal fees upfront to neutralize fee shock.
2. **In-Cart Milestone Progress Bar**: 'You are ₹65 away from FREE DELIVERY!' (Drives AOV while eliminating fee resistance).
3. **Eco-Saver Flexible Pacing**: Allows users to choose between Priority (₹35 / 25m) and Eco-Saver Pooled Delivery (₹0 / 40m).

---
## 7. A/B Testing Measurement Framework
- **Primary Metric**: Checkout-to-Order Conversion Rate (Baseline 29.37%, Target: +3.5% lift to 32.87%).
- **Power Analysis**: Alpha = 0.05, Power = 0.80, MDE = 2.5%, Required Sample Size = ~25,000 sessions (21-day runtime).
- **Guardrails**: Contribution margin per order, courier fulfillment SLA latency.